In [1]:
!pip install mlflow boto3 awscli optuna imbalanced-learn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# !aws configure  # skipped: interactive; not needed with local MLflow

In [3]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("file:./mlruns")

In [4]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location=('file:///C:/Users/HAI/Downloads/Youtube sentiment '
 'analysis/notebooks/mlruns/775459616684597847'), creation_time=1787582478269, experiment_id='775459616684597847', last_update_time=1787582478269, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}>

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import RandomOverSampler
import mlflow
import mlflow.sklearn
import optuna


C:\Users\HAI\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import os
df = pd.read_csv(('tweets_preprocessing.csv' if os.path.exists('tweets_preprocessing.csv') else '/content/tweets_preprocessing.csv' if os.path.exists('/content/tweets_preprocessing.csv') else '../tweets_preprocessing.csv')).dropna()
df.shape

(54036, 14)

In [7]:
# Step 1: (Optional) Remapping - skipped since not strictly needed for Random Forest

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for Random Forest
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)  # Number of trees in the forest
    max_depth = trial.suggest_int('max_depth', 3, 20)  # Maximum depth of the tree
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)  # Minimum samples required to split a node
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)  # Minimum samples required at a leaf node

    # RandomForestClassifier setup
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                   min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
                                   random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for Random Forest, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_rf, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = RandomForestClassifier(n_estimators=best_params['n_estimators'],
                                        max_depth=best_params['max_depth'],
                                        min_samples_split=best_params['min_samples_split'],
                                        min_samples_leaf=best_params['min_samples_leaf'],
                                        random_state=42)

    # Log the best model with MLflow, passing the algo_name as "RandomForest"
    log_mlflow("RandomForest", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for Random Forest
run_optuna_experiment()


[I 2026-08-24 21:32:45,321] A new study created in memory with name: no-name-3aa42223-d28d-4f80-bf19-fcab18096e8d


[I 2026-08-24 21:32:57,398] Trial 0 finished with value: 0.6260640767683021 and parameters: {'n_estimators': 268, 'max_depth': 17, 'min_samples_split': 11, 'min_samples_leaf': 13}. Best is trial 0 with value: 0.6260640767683021.


[I 2026-08-24 21:33:06,229] Trial 1 finished with value: 0.6036217303822937 and parameters: {'n_estimators': 273, 'max_depth': 11, 'min_samples_split': 12, 'min_samples_leaf': 19}. Best is trial 0 with value: 0.6260640767683021.


[I 2026-08-24 21:33:17,732] Trial 2 finished with value: 0.6262962389722954 and parameters: {'n_estimators': 259, 'max_depth': 17, 'min_samples_split': 9, 'min_samples_leaf': 17}. Best is trial 2 with value: 0.6262962389722954.


[I 2026-08-24 21:33:19,706] Trial 3 finished with value: 0.5643863179074446 and parameters: {'n_estimators': 130, 'max_depth': 3, 'min_samples_split': 7, 'min_samples_leaf': 9}. Best is trial 2 with value: 0.6262962389722954.


...

[I 2026-08-24 21:35:04,646] Trial 26 finished with value: 0.6105865965020895 and parameters: {'n_estimators': 300, 'max_depth': 13, 'min_samples_split': 9, 'min_samples_leaf': 20}. Best is trial 22 with value: 0.6399938090078935.


[I 2026-08-24 21:35:09,877] Trial 27 finished with value: 0.6361244389413403 and parameters: {'n_estimators': 102, 'max_depth': 19, 'min_samples_split': 5, 'min_samples_leaf': 12}. Best is trial 22 with value: 0.6399938090078935.


[I 2026-08-24 21:35:16,662] Trial 28 finished with value: 0.6235876799257081 and parameters: {'n_estimators': 145, 'max_depth': 16, 'min_samples_split': 3, 'min_samples_leaf': 18}. Best is trial 22 with value: 0.6399938090078935.


[I 2026-08-24 21:35:19,861] Trial 29 finished with value: 0.6256771397616468 and parameters: {'n_estimators': 73, 'max_depth': 17, 'min_samples_split': 4, 'min_samples_leaf': 14}. Best is trial 22 with value: 0.6399938090078935.


2026/08/24 21:35:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
